# 03 · QAOA com Qiskit: quanto o algoritmo realmente acerta?

**Pergunta-guia:** o QAOA encontra a carteira ótima do QUBO do notebook 02? E, mais importante, com que **probabilidade**?

Há uma distinção que este notebook torna explícita:

- `res.x` é o resultado que o solver devolve. Pelo `MinimumEigenOptimizer`, é a amostra de **menor energia** entre as obtidas, não necessariamente a mais provável.
- A **distribuição de amostras** mostra o que o circuito produz de fato.

Medimos as duas coisas, e comparamos com dois chutes uniformes: 1/16 para o ótimo e 6/16 para carteiras viáveis, sem nenhum algoritmo.

**Ambiente:** kernel `venv-quantum` (Qiskit 2.6.2, qiskit-aer 0.18.2, qiskit-algorithms 0.4.0, qiskit-optimization 0.7.0). Para abrir no Jupyter: `venv-quantum/bin/python -m ipykernel install --user --name venv-quantum`.

## 1. Setup: mesmo problema do notebook 02

Reconstruímos μ, Σ (treino 2023–2024, anualizados) e o QUBO com os mesmos parâmetros: $q = 0{,}5$, $B = 2$, $\lambda = 1$. Depois conferimos que a função objetivo é a mesma do notebook 02.

In [1]:
import warnings
warnings.filterwarnings('ignore', message='.*SparseEfficiencyWarning.*')
warnings.filterwarnings('ignore', category=Warning, module='scipy.sparse')

import numpy as np
import pandas as pd
from itertools import product

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import generate_preset_pass_manager

np.set_printoptions(precision=4, suppress=True)

# Dados e período de treino (mesmos do notebook 02)
df = pd.read_csv('../data/precos_b3.csv', header=[0, 1], index_col=0, skiprows=[2], parse_dates=True)
close = df['Close'].apply(pd.to_numeric, errors='coerce').dropna()
close.index = pd.to_datetime(close.index)
ativos = list(close.columns)
n = len(ativos)

ret = close.loc['2023-01-01':'2024-12-31'].pct_change().dropna()
mu = (ret.mean() * 252).values
Sigma = (ret.cov() * 252).values

B, q, lam = 2, 0.5, 1.0

def build_qubo(lam):
    ones = np.ones(n)
    Q = q * Sigma + lam * np.outer(ones, ones)
    c = -mu - 2 * lam * B * ones
    d = lam * B ** 2
    return Q, c, d

def build_qp(lam):
    Q, c, d = build_qubo(lam)
    qp = QuadraticProgram(f'carteira_b3_lam{lam}')
    for a in ativos:
        qp.binary_var(name=a)
    qp.minimize(constant=d, linear=c, quadratic=Q)
    return qp

Q, c, d = build_qubo(lam)
qp = build_qp(lam)
todos = np.array(list(product([0, 1], repeat=n)))
f = lambda x: x @ Q @ x + c @ x + d

erro = max(abs(qp.objective.evaluate(x) - f(x)) for x in todos)
print(f'Erro entre QuadraticProgram e forma QUBO: {erro:.2e}')
assert erro < 1e-12

# Ótimo de referência por força bruta
x_ot, v_ot = min(((x, f(x)) for x in todos), key=lambda t: t[1])
print('Ótimo (força bruta):', x_ot, '| viável:', x_ot.sum() == B, '| f =', round(v_ot, 4))
print('Ativos:', [a for a, s in zip(ativos, x_ot) if s])

Erro entre QuadraticProgram e forma QUBO: 2.22e-16
Ótimo (força bruta): [1 0 1 0] | viável: True | f = -0.4465
Ativos: ['BBAS3.SA', 'KLBN4.SA']


## 2. Conferindo que o Hamiltoniano de Qiskit é o mesmo QUBO

O `QuadraticProgram.to_ising()` devolve um operador de Pauli e uma constante. Se a conversão estiver certa, a energia de cada basis state $|k\rangle$ mais a constante deve ser igual a $f(x)$, com $x$ a bitstring correspondente.

Convenção do Qiskit: a variável $i$ corresponde ao qubit $i$, e a ordem de bits é little-endian (o qubit 0 é o bit menos significativo). Assim, $k = \sum_i x_i 2^i$.

In [2]:
op, offset = qp.to_ising()
print('Número de qubits no Hamiltoniano:', op.num_qubits)
print('Termos de Pauli:', len(op.paulis))
print('Constante (offset):', round(float(offset), 4))

diag = np.real(np.diag(op.to_matrix()))   # H é diagonal na base computacional

erro_ising = 0.0
for x in todos:
    k = int(sum(int(xi) << i for i, xi in enumerate(x)))
    erro_ising = max(erro_ising, abs(diag[k] + offset - f(x)))
print(f'\nErro máximo entre H (Pauli) + offset e f(x): {erro_ising:.2e}')
assert erro_ising < 1e-10

Número de qubits no Hamiltoniano: 4
Termos de Pauli: 10
Constante (offset): 0.7859

Erro máximo entre H (Pauli) + offset e f(x): 1.33e-15


## 3. Referências clássicas e chutes uniformes

Antes de rodar o QAOA, medimos o que cada alternativa entregaria:

- **Solver exato** (`NumPyMinimumEigensolver`): resolve o Hamiltoniano diagonalizando a matriz. Serve para checar a formulação quântica, e não é um método quântico.
- **Chute uniforme:** sortear um dos 16 vetores ao acaso. Acerta o ótimo com probabilidade 1/16 e produz uma carteira viável com probabilidade 6/16.

In [3]:
exato = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)
print('Solver exato:', exato.x, '| fval =', round(exato.fval, 4), '|', exato.status.name)

p_uniforme_otimo = 1 / 2 ** n
p_uniforme_viavel = sum(1 for x in todos if x.sum() == B) / 2 ** n
print(f'\nChute uniforme: P(ótimo) = {p_uniforme_otimo:.4f} | P(viável) = {p_uniforme_viavel:.4f}')

Solver exato: [1. 0. 1. 0.] | fval = -0.4465 | SUCCESS

Chute uniforme: P(ótimo) = 0.0625 | P(viável) = 0.3750


## 4. Uma execução do QAOA, lida em detalhe

Parâmetros: $p = 2$ camadas, com $2p = 4$ ângulos no total ($\gamma$ e $\beta$ em cada camada), otimizador COBYLA com 300 iterações, simulador de vetor de estado com 4096 shots.

Para cada execução, medimos:

- `x do solver`: o que o `MinimumEigenOptimizer` devolveu.
- `P(ótimo)`: probabilidade total das amostras iguais ao ótimo de referência.
- `P(viável)`: probabilidade total das amostras com exatamente $B$ ativos.

In [4]:
def metricas(res):
    p_opt = sum(s.probability for s in res.samples if np.array_equal(s.x, x_ot))
    p_viav = sum(s.probability for s in res.samples if s.x.sum() == B)
    return p_opt, p_viav

def rodar_qaoa(p, seed, lam_t=None, sampler=None, transpiler=None):
    qp_t = qp if lam_t is None else build_qp(lam_t)
    np.random.seed(seed)
    if sampler is None:
        sampler = StatevectorSampler(default_shots=4096, seed=seed)
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=p, transpiler=transpiler)
    res = MinimumEigenOptimizer(qaoa).solve(qp_t)
    p_opt, p_viav = metricas(res)
    return {
        'p': p,
        'seed': seed,
        'x do solver': res.x,
        'acertou solver': bool(np.array_equal(res.x, x_ot)),
        'P(ótimo)': p_opt,
        'P(viável)': p_viav,
        'avaliações': res.min_eigen_solver_result.cost_function_evals,
        'res': res,
    }

exec_1 = rodar_qaoa(p=2, seed=0)
print('x do solver:', exec_1['x do solver'], '| acertou:', exec_1['acertou solver'])
print(f"P(ótimo) = {exec_1['P(ótimo)']:.4f}  (chute uniforme: {p_uniforme_otimo:.4f})")
print(f"P(viável) = {exec_1['P(viável)']:.4f}  (chute uniforme: {p_uniforme_viavel:.4f})")
print('\nAmostras mais prováveis:')
for s in sorted(exec_1['res'].samples, key=lambda s: -s.probability)[:6]:
    print(' ', s.x.astype(int), f'fval={s.fval:+.4f}', f'prob={s.probability:.4f}')

x do solver: [1. 0. 1. 0.] | acertou: True
P(ótimo) = 0.0151  (chute uniforme: 0.0625)
P(viável) = 0.0986  (chute uniforme: 0.3750)

Amostras mais prováveis:
  [1 1 0 0] fval=-0.4194 prob=0.0205
  [0 0 1 1] fval=-0.0057 prob=0.0184
  [0 1 1 0] fval=-0.3473 prob=0.0180
  [1 0 1 0] fval=-0.4465 prob=0.0151
  [0 1 0 1] fval=+0.0008 prob=0.0136
  [1 0 0 1] fval=-0.1017 prob=0.0131


Nesta execução, o solver acerta o ótimo, mas a probabilidade do ótimo é baixa. Isso porque o `MinimumEigenOptimizer` escolhe a amostra de menor energia dentre as obtidas. Se o circuito espalha a massa de probabilidade, a resposta "certa" pode aparecer sem ser a mais provável. Quem quer usar o QAOA como amostrador precisa olhar a distribuição, e não só `res.x`.

## 5. Varredura de profundidade e sementes

Variamos $p \in \{1, 2, 3\}$ e 5 sementes para cada um. O resultado mostra se mais camadas melhoram a probabilidade e se o desempenho depende da semente.

In [5]:
linhas = []
for p in [1, 2, 3]:
    for seed in range(5):
        r = rodar_qaoa(p=p, seed=seed)
        linhas.append({k: r[k] for k in ['p', 'seed', 'acertou solver', 'P(ótimo)', 'P(viável)', 'avaliações']})

sweep = pd.DataFrame(linhas)
print(sweep.round(4).to_string(index=False))

resumo = sweep.groupby('p').agg(
    acertos_solver=('acertou solver', 'mean'),
    P_otimo_media=('P(ótimo)', 'mean'),
    P_otimo_desvio=('P(ótimo)', 'std'),
    P_viavel_media=('P(viável)', 'mean'),
).round(4)
print(f"\nChute uniforme: P(ótimo) = {p_uniforme_otimo:.4f} | P(viável) = {p_uniforme_viavel:.4f}")
resumo

 p  seed  acertou solver  P(ótimo)  P(viável)  avaliações
 1     0            True    0.0249     0.0779          38
 1     1            True    0.0019     0.0111          34
 1     2            True    0.0081     0.0917          48
 1     3            True    0.0039     0.0614          33
 1     4            True    0.0204     0.0798          38
 2     0            True    0.0621     0.1230          52
 2     1            True    0.0031     0.0767          61
 2     2            True    0.0253     0.0818          45
 2     3            True    0.0073     0.0891          51
 2     4            True    0.0151     0.0908          55
 3     0            True    0.0574     0.1346          85
 3     1            True    0.0662     0.1530          55
 3     2            True    0.0516     0.1504          63
 3     3            True    0.0306     0.1122         114
 3     4            True    0.0182     0.0424          80

Chute uniforme: P(ótimo) = 0.0625 | P(viável) = 0.3750


,acertos_solver,P_otimo_media,P_otimo_desvio,P_viavel_media
p,,,,
1,1.0,0.0118,0.0102,0.0644
2,1.0,0.0226,0.0237,0.0923
3,1.0,0.0448,0.0198,0.1185


## 6. Penalidade $\lambda$ sob o QAOA

No notebook 02, $\lambda$ = 0,1 deixava o ótimo do QUBO inviável. Aqui vemos o que isso faz com o circuito, em simulação ideal com $p = 2$ e 3 sementes.

Leitura da tabela: com $\lambda = 0{,}1$, o solver deixa de acertar $x^*$ (esperado, porque o QUBO muda e $x^*$ é o de $\lambda = 1$), e `P(viável)` cai de cerca de 8,6% para 5,2%. O efeito da penalidade no circuito existe, mas é pequeno nesta faixa. O dado que domina a tabela é outro: `P(viável)` fica abaixo do chute uniforme de 37,5% em todos os casos.

In [6]:
linhas_lam = []
for lam_t in [0.1, 1.0, 10.0]:
    for seed in range(3):
        r = rodar_qaoa(p=2, seed=seed, lam_t=lam_t)
        linhas_lam.append({'λ': lam_t, 'seed': seed, 'acertou solver': r['acertou solver'],
                           'P(ótimo)': r['P(ótimo)'], 'P(viável)': r['P(viável)']})

pen = pd.DataFrame(linhas_lam)
pen.groupby('λ').agg(P_viavel=('P(viável)', 'mean'), P_otimo=('P(ótimo)', 'mean'),
                     acertos_solver=('acertou solver', 'mean')).round(4)

,P_viavel,P_otimo,acertos_solver
λ,,,
0.1,0.0390,0.0249,0.0
1.0,0.0723,0.0191,1.0
10.0,0.1039,0.0202,1.0


**Atenção:** a referência $x^*$ é o ótimo do QUBO com $\lambda$ = 1, usado em todas as linhas. Com $\lambda = 0{,}1$, o QUBO pode ter outro mínimo, então `P(ótimo)` mede proximidade a este $x^*$, e não ao mínimo daquela formulação. Use `P(viável)` para comparar $\lambda$s.

## 7. Efeito de ruído (modelo ilustrativo)

Com o simulador Aer, aplicamos um modelo de ruído de despolarização. As taxas abaixo são **ilustrativas**, escolhidas para mostrar o efeito, e não foram calibradas em nenhum hardware real. O transpilador é necessário para o Aer reconhecer as portas do circuito do QAOA.

In [7]:
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ['cx'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ['ry', 'rx', 'rz', 'h'])

pm = generate_preset_pass_manager(backend=AerSimulator(), optimization_level=1)

linhas_ruido = []
for p in [1, 2]:
    for seed in range(3):
        aer = AerSamplerV2(default_shots=4096, seed=seed,
                           options={'backend_options': {'noise_model': nm}})
        r = rodar_qaoa(p=p, seed=seed, sampler=aer, transpiler=pm)
        linhas_ruido.append({'p': p, 'seed': seed, 'acertou solver': r['acertou solver'],
                             'P(ótimo)': r['P(ótimo)'], 'P(viável)': r['P(viável)']})

ruido = pd.DataFrame(linhas_ruido)
print(ruido.round(4).to_string(index=False))
print(f"\nChute uniforme: P(ótimo) = {p_uniforme_otimo:.4f} | P(viável) = {p_uniforme_viavel:.4f}")
ruido.groupby('p').agg(P_otimo=('P(ótimo)', 'mean'), P_viavel=('P(viável)', 'mean'),
                       acertos_solver=('acertou solver', 'mean')).round(4)

 p  seed  acertou solver  P(ótimo)  P(viável)
 1     0            True    0.0161     0.0380
 1     1            True    0.0155     0.0857
 1     2            True    0.0028     0.0582
 2     0            True    0.0188     0.1498
 2     1            True    0.0220     0.0747
 2     2            True    0.0258     0.0972

Chute uniforme: P(ótimo) = 0.0625 | P(viável) = 0.3750


,P_otimo,P_viavel,acertos_solver
p,,,
1,0.0115,0.0606,1.0
2,0.0222,0.1072,1.0


## 8. O que este notebook mostra

1. **A formulação quântica bate com a clássica.** O Hamiltoniano de Qiskit reproduz $f(x)$ em todas as 16 bitstrings (seção 2), e o solver exato devolve o mesmo ótimo (seção 3).
2. **Acertar o ótimo não é o mesmo que amostrar bem.** O `res.x` do solver pode estar certo enquanto a probabilidade do ótimo é menor que a de um chute uniforme. Por isso reportamos `P(ótimo)` e `P(viável)`.
3. **Mais camadas ajudam pouco e a variância é alta.** Na seção 5, `P(ótimo)` vai de cerca de 1,2% (p=1) para 2,3% (p=2) e 2,5% (p=3), e `P(viável)` não cresce de forma monotônica (7,5%, 13,5%, 9,7%). A variação entre sementes é maior que a diferença entre profundidades, então a média sozinha engana.
4. **A penalidade $\lambda$ tem efeito pequeno nesta faixa.** Na seção 6, $\lambda = 0{,}1$ reduz `P(viável)` de ~8,6% para ~5,2%. Com 3 sementes, não dá para afirmar o efeito com segurança.
5. **Ruído muda a distribuição, sem padrão claro.** Na seção 7, `P(viável)` médio em $p = 2$ cai de 13,5% (ideal) para 9,5% (ruidoso), mas a dispersão é grande (uma semente chega a 0,2%). As taxas são ilustrativas e o resultado não é previsão para hardware real.
6. **O mais importante:** em todas as configurações, `P(viável)` fica abaixo do chute uniforme (37,5%): médias entre 5% e 13%, e valores individuais entre 0,2% e 16%. `P(ótimo)` também fica abaixo de 6,25% em média (médias entre 1,2% e 2,8%). Com essa formulação e esses parâmetros, o QAOA como amostrador é pior que sortear ao acaso. O solver "acerta" em 100% das execuções apenas porque escolhe a menor energia entre as amostras, e isso esconde a qualidade da distribuição.

## 9. Limites

- Com 4 qubits, a força bruta resolve em microssegundos. Este notebook avalia a **mecânica** do QAOA, e não uma vantagem sobre o clássico.
- Usamos apenas o período de treino. O teste fora da amostra (2025) fica para um passo posterior, comparando a carteira quântica com pesos iguais e Markowitz contínuo.
- O modelo de ruído é ilustrativo. Calibrar a partir de um dispositivo real exige outra etapa.
- As probabilidades são estimativas por amostragem (4096 shots). O erro estatístico de cada `P(ótimo)` é da ordem de $\sqrt{p(1-p)/4096}$, cerca de 0,2 pontos percentuais em $p \approx 0{,}02$.